# Belief Revision

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Function:** Evidence integration & updating

Can the model update its answer when given new evidence that changes the correct conclusion?
Tests whether self-generated notes about what changed help overcome anchoring bias.

**Difficulty grid:** reasoning depth × contradiction strength × 3 seeds = 27 items


In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)


In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

def extract_number(response: str) -> str:
    """Extract a number from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean): return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

def extract_category(response: str) -> str:
    """Extract a category letter (A-D) from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        m = re.search(r'\b([A-D])\b', line)
        if m and len(line) < 30: return m.group(1)
    for line in reversed(lines):
        m = re.search(r'([A-D])', line)
        if m: return m.group(1)
    return ''


In [ ]:
# === Belief revision: constraint satisfaction with updates ===
# Present initial constraints → answer A, then add constraints → answer changes to B.

def gen_simple(seed, strength):
    """Simple: find X given numeric constraints."""
    rng = random.Random(seed)
    target_initial = rng.randint(10, 50)
    # Initial constraints that point to target_initial
    lo = target_initial - rng.randint(1, 3)
    hi = target_initial + rng.randint(1, 3)
    parity = 'even' if target_initial % 2 == 0 else 'odd'

    initial = f"Find X where: X > {lo}, X < {hi}, X is {parity}."

    # New constraint changes the answer
    if strength == 'weak':
        new_target = target_initial + (2 if target_initial % 2 == 0 else 2)
        update = f"Additional constraint: X > {target_initial}."
    elif strength == 'moderate':
        new_target = rng.randint(60, 90)
        new_parity = 'even' if new_target % 2 == 0 else 'odd'
        update = f"CORRECTION: The range was wrong. X > {new_target - 2}, X < {new_target + 2}, X is {new_parity}."
    else:  # strong
        new_target = rng.randint(60, 90)
        new_parity = 'even' if new_target % 2 == 0 else 'odd'
        update = f"DISREGARD previous constraints. New constraints: X > {new_target - 1}, X < {new_target + 1}, X is {new_parity}."

    return initial, update, str(target_initial), str(new_target), f'simple_{strength}'

def gen_chain(seed, strength):
    """Causal chain: A causes B causes C, then A changes."""
    rng = random.Random(seed)
    people = ['Alice','Bob','Carol','Dave','Eve','Frank']
    p = rng.sample(people, 4)

    initial = f"{p[0]} gave a book to {p[1]}. {p[1]} always shares books with {p[2]}. Who has the book?"
    initial_answer = p[2]

    if strength == 'weak':
        update = f"Actually, {p[1]} kept the book this time."
        new_answer = p[1]
    elif strength == 'moderate':
        update = f"Correction: {p[0]} gave the book to {p[3]}, not {p[1]}. {p[3]} always shares with {p[2]}."
        new_answer = p[2]  # same answer but different path
    else:
        update = f"Correction: {p[0]} gave the book to {p[3]}, not {p[1]}. {p[3]} never shares books."
        new_answer = p[3]

    return initial, update, initial_answer, new_answer, f'chain_{strength}'

def gen_nested(seed, strength):
    """Nested reasoning: if-then chains that get overridden."""
    rng = random.Random(seed)
    colors = ['red','blue','green','yellow']
    c = rng.sample(colors, 4)

    initial = (f"Rule 1: If the light is {c[0]}, press button A. "
               f"Rule 2: If the light is {c[1]}, press button B. "
               f"Rule 3: If the light is {c[2]}, press button C. "
               f"The light is {c[0]}. What button do you press?")
    initial_answer = 'A'

    if strength == 'weak':
        update = f"Addendum: Rule 1 is suspended today. Default to button D."
        new_answer = 'D'
    elif strength == 'moderate':
        update = f"Correction: The light is actually {c[1]}, not {c[0]}."
        new_answer = 'B'
    else:
        update = f"OVERRIDE: The light is {c[2]}. Also, Rule 3 now maps to button A instead of C."
        new_answer = 'A'

    return initial, update, initial_answer, new_answer, f'nested_{strength}'

BELIEF_FNS = {'simple': gen_simple, 'chain': gen_chain, 'nested': gen_nested}
STRENGTHS = ['weak', 'moderate', 'strong']


In [ ]:
# === Dataset ===
SEEDS = 3
rows = []; tid = 0

for complexity, bfn in BELIEF_FNS.items():
    for strength in STRENGTHS:
        for seed in range(SEEDS):
            initial, update, old_answer, new_answer, desc = bfn(seed * 100, strength)
            # Material = initial + update (all evidence)
            material = f"INITIAL EVIDENCE:\n{initial}\n\nNEW EVIDENCE:\n{update}"
            label = f'{complexity}_{strength}'
            rows.append({
                'task_id': tid, 'seed': seed, 'complexity': complexity,
                'evidence': strength, 'difficulty_label': label,
                'material': material, 'test_input': initial,
                'expected': new_answer.lower(), 'item_desc': desc,
                'initial_evidence': initial, 'update_evidence': update,
                'old_answer': old_answer,
            })
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id','difficulty_label','expected','item_desc']].to_string(index=False))


In [ ]:
DB_PATH = 'belief_revision_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        complexity TEXT, evidence TEXT, difficulty_label TEXT,
        seed INTEGER, item_desc TEXT,
        test_input TEXT, expected TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')


In [ ]:
PRE_P = """Consider ALL of the following evidence:

{material}

Based on ALL the evidence (including any corrections or updates), what is the answer?

Reply with ONLY the answer (a name, letter, or number). Nothing else."""

STUDY_P = """Consider the following evidence that includes an update/correction:

{material}

Analyze what changed:
1. What did the initial evidence suggest?
2. What does the new evidence change?
3. How does this affect the conclusion?
4. What is the correct answer considering ALL evidence?

Write a clear analysis."""

POST_P = """You previously analyzed some evidence with an update and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here is the full evidence again:
{material}

Based on ALL the evidence and your analysis, what is the answer?

Reply with ONLY the answer (a name, letter, or number). Nothing else."""


## Evaluation


In [ ]:
@kbench.task(name='belief_revision',
             description='Pre/post learning belief revision with contradicting evidence')
def belief_revision(llm, material: str, test_input: str, expected: str,
                    complexity: str, evidence: str, difficulty_label: str,
                    seed: int, item_desc: str, initial_evidence: str,
                    update_evidence: str, old_answer: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{complexity}_{evidence}_{seed}'

    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material))
    pre_time = time.time() - t0
    pre_extracted = extract_short_answer(pre_raw)
    pre_correct = pre_extracted == expected

    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material))
    post_time = time.time() - t0
    post_extracted = extract_short_answer(post_raw)
    post_correct = post_extracted == expected

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,complexity,evidence,difficulty_label,
               seed,item_desc,test_input,expected,pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts,model_name,tid,complexity,evidence,difficulty_label,int(seed),item_desc,
             test_input,expected,pre_raw,pre_extracted,int(pre_correct),
             notes,post_raw,post_extracted,int(post_correct),pre_time,study_time,post_time))
        db.commit()
    except Exception as e: print(f'  [SQLite warn] {e}')

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{difficulty_label:16s}] expected="{expected}"  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

runs = belief_revision.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                 n_jobs=1, timeout=3600, max_attempts=1)
print(f'\nDone: {len(runs.as_dataframe())} items')


## Results


In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY task_id', db)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:12s}] "{row["test_input"]}" -> "{row["expected"]}"  '
          f'pre="{row["pre_extracted"]}"({b})  post="{row["post_extracted"]}"({l}){delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')


In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')


In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('belief_revision_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench — {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')
